# Project 2: YouTube Advertising Budget Optimizer

**Business question:** *We have Rs. 1,000,000 to spend on YouTube advertising next month. How should we split it across our YouTube campaign types to get the most revenue?*

**Scope:** I chose a single platform (YouTube), as my teacher allowed, so there is no Facebook / Instagram / Google / TikTok split. The budget recommendation is made **across YouTube campaign types** instead (In-Stream, Shorts, Discovery, Bumper, Retargeting).

**Data note:** The data is **simulated** (the project brief asks for simulated historical performance). It is generated with a fixed random seed so results are repeatable. Real campaign performance would differ.

## Step 1 · Generate the data and keep a backup

In [ ]:
import os, shutil, time, csv, warnings
from collections import defaultdict
from multiprocessing import Pool
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
os.makedirs("submission", exist_ok=True)

# ---------- Assumptions that drive the simulation (change them and re-run) ----------
# w = share of records, cpm = cost per 1000 impressions (Rs), ctr = click-through rate,
# cvr = conversion rate (conversions / clicks), aov = average order value (Rs),
# sat = saturation: how fast cost per impression rises when the daily budget is pushed up
CAMPAIGNS = {
    "In-Stream Skippable": dict(w=0.30, cpm=420, ctr=0.012, cvr=0.035, aov=4200, sat=0.25),
    "Shorts Ads":          dict(w=0.25, cpm=260, ctr=0.009, cvr=0.020, aov=3600, sat=0.10),
    "Discovery Ads":       dict(w=0.20, cpm=380, ctr=0.015, cvr=0.040, aov=4600, sat=0.30),
    "Bumper Ads":          dict(w=0.15, cpm=300, ctr=0.006, cvr=0.015, aov=3200, sat=0.15),
    "Retargeting":         dict(w=0.10, cpm=650, ctr=0.030, cvr=0.090, aov=5200, sat=0.60),
}
AGES    = ["18-24", "25-34", "35-44", "45-54", "55+"]
AGE_W   = [0.28, 0.34, 0.20, 0.11, 0.07];  AGE_CVR = [0.8, 1.2, 1.1, 0.9, 0.7]
GENDERS = ["Male", "Female"];              GEN_W = [0.55, 0.45];  GEN_CVR = [0.95, 1.05]
CITIES  = ["Karachi", "Lahore", "Islamabad", "Rawalpindi", "Faisalabad", "Multan", "Peshawar"]
CITY_W  = [0.26, 0.24, 0.14, 0.10, 0.10, 0.09, 0.07]
CITY_CVR = [1.0, 1.05, 1.25, 1.0, 0.85, 0.8, 0.8]
DEVICES = ["Mobile", "Desktop", "Tablet", "Smart TV"]
DEV_W   = [0.72, 0.15, 0.05, 0.08];        DEV_CVR = [0.95, 1.25, 0.9, 0.7]
N_DAYS  = 90
START   = pd.Timestamp("2025-01-01")

def make_ads_data(n, seed=42):
    """One record = one slice of a campaign on one day (age x gender x city x device)."""
    rng = np.random.default_rng(seed)
    names = list(CAMPAIGNS); P = [CAMPAIGNS[c] for c in names]
    c  = rng.choice(len(names),   n, p=[p["w"] for p in P])
    d  = rng.integers(0, N_DAYS, n)
    a  = rng.choice(len(AGES),    n, p=AGE_W)
    g  = rng.choice(len(GENDERS), n, p=GEN_W)
    ci = rng.choice(len(CITIES),  n, p=CITY_W)
    dv = rng.choice(len(DEVICES), n, p=DEV_W)

    # Daily budget level per campaign (same pattern for every dataset size)
    day_mult = np.random.default_rng(7).lognormal(0, 0.5, (N_DAYS, len(names)))
    m   = day_mult[d, c]
    cpm = np.array([p["cpm"] for p in P])[c]
    sat = np.array([p["sat"] for p in P])[c]
    ctr = np.array([p["ctr"] for p in P])[c]
    cvr = np.array([p["cvr"] for p in P])[c]
    aov = np.array([p["aov"] for p in P])[c]

    spend       = rng.lognormal(np.log(400), 0.35, n) * m
    # higher daily budget -> higher cost per impression (diminishing returns)
    impressions = (spend / (cpm * m**sat / 1000)).astype(np.int64)
    clicks      = rng.binomial(impressions, np.clip(ctr * rng.normal(1, .1, n), 0.001, 0.2))
    p_conv      = np.clip(cvr * np.array(AGE_CVR)[a] * np.array(GEN_CVR)[g]
                          * np.array(CITY_CVR)[ci] * np.array(DEV_CVR)[dv], 0.001, 0.5)
    conversions = rng.binomial(clicks, p_conv)
    revenue     = conversions * aov * rng.lognormal(0, 0.15, n)

    return pd.DataFrame({
        "date": (START + pd.to_timedelta(d, unit="D")).astype("datetime64[ns]"),
        "campaign": pd.Categorical.from_codes(c, names),
        "platform": "YouTube",
        "impressions": impressions, "clicks": clicks,
        "spend": spend.round(2), "conversions": conversions, "revenue": revenue.round(2),
        "age": pd.Categorical.from_codes(a, AGES),
        "gender": pd.Categorical.from_codes(g, GENDERS),
        "city": pd.Categorical.from_codes(ci, CITIES),
        "device": pd.Categorical.from_codes(dv, DEVICES),
    })

# Base dataset: 10,000 records
raw = make_ads_data(10_000)
raw.to_csv("youtube_ads_10k.csv", index=False)
shutil.copy("youtube_ads_10k.csv", "ORIGINAL_backup_youtube_ads_10k.csv")   # untouched copy
df = raw.copy()
print("Generated:", df.shape)
df.head()

## Step 2 · What is one record, and what does each column mean?

In [ ]:
meanings = {
    "date": "Day the ads ran",
    "campaign": "YouTube campaign type",
    "platform": "Advertising platform (always YouTube in my project)",
    "impressions": "Number of times the ad was shown",
    "clicks": "Number of clicks on the ad",
    "spend": "Money spent (Rs.)",
    "conversions": "Purchases / sign-ups that came from the ad",
    "revenue": "Revenue from those conversions (Rs.)",
    "age": "Age group of the audience slice",
    "gender": "Gender of the audience slice",
    "city": "City of the audience slice",
    "device": "Device the ad was watched on",
}
def kind(s):
    if pd.api.types.is_numeric_dtype(s):          return "Numerical"
    if pd.api.types.is_datetime64_any_dtype(s):   return "Date/time"
    return "Text/Categorical"

data_dictionary = pd.DataFrame({
    "column": df.columns,
    "dtype_as_loaded": [str(t) for t in df.dtypes],
    "type_guess": [kind(df[c]) for c in df.columns],
    "example_value": [df[c].dropna().iloc[0] for c in df.columns],
    "meaning": [meanings.get(c, "TODO: describe this column") for c in df.columns],
})
data_dictionary.to_csv("submission/data_dictionary.csv", index=False)
data_dictionary

**One record** = the ads of one campaign type on one day, for one audience slice (age group, gender, city, device).

## Step 3 · Inspect before cleaning

Because the data is simulated it should be clean. The checks below are still run so the process is documented, and they would catch problems in real exported campaign data.

In [ ]:
print("Rows, columns:", df.shape)
display(df.dtypes)
display(df.isna().sum())
print("Duplicate rows:", df.duplicated().sum())

# Impossible values in ad data
checks = {
    "negative spend or revenue":       ((df.spend < 0) | (df.revenue < 0)).sum(),
    "clicks > impressions":            (df.clicks > df.impressions).sum(),
    "conversions > clicks":            (df.conversions > df.clicks).sum(),
    "zero impressions":                (df.impressions == 0).sum(),
    "zero spend":                      (df.spend == 0).sum(),
    "zero clicks (CPC undefined)":     (df.clicks == 0).sum(),
    "zero conversions (CPA undefined)": (df.conversions == 0).sum(),
}
display(pd.Series(checks, name="records"))
display(df.describe().T)

## Step 4 · Clean (every change is logged)

Rule used here: **do not silently delete rows.** Impossible values become *missing* and are reported.
Rows with zero clicks or zero conversions are **kept**: they are real outcomes, and the ratio metrics (CPC, CPA) are always calculated on **totals**, so a zero does not cause a divide-by-zero.

In [ ]:
log = []
def note(step, n, detail=""):
    log.append({"step": step, "rows_or_cells_affected": int(n), "detail": detail})

# 1) Impossible values -> NaN (logged)
for col in ["spend", "revenue", "impressions", "clicks", "conversions"]:
    bad = (df[col] < 0).sum()
    df[col] = df[col].mask(df[col] < 0)
    note(f"negative {col} set to NaN", bad)

bad = (df.clicks > df.impressions).sum()
df.loc[df.clicks > df.impressions, "clicks"] = np.nan
note("clicks > impressions -> clicks NaN", bad)

bad = (df.conversions > df.clicks).sum()
df.loc[df.conversions > df.clicks, "conversions"] = np.nan
note("conversions > clicks -> conversions NaN", bad)

# 2) Standardise text categories
for c in ["campaign", "platform", "age", "gender", "city", "device"]:
    df[c] = df[c].astype("string").str.strip()
note("trim whitespace in text columns", 0)

# 3) Duplicates
n = df.duplicated().sum()
df = df.drop_duplicates()
note("drop exact duplicate rows", n)

cleaning_log = pd.DataFrame(log)
cleaning_log.to_csv("submission/cleaning_log.csv", index=False)
display(cleaning_log)
print("Rows after cleaning:", len(df))
print("Missing values after cleaning:")
print(df.isna().sum()[lambda s: s > 0])

# Back to categorical (faster group-bys)
for c in ["campaign", "age", "gender", "city", "device"]:
    df[c] = df[c].astype("category")

## Step 5 · Marketing KPIs

| KPI | Formula | Meaning |
|---|---|---|
| CTR | clicks ÷ impressions | How many viewers click |
| CPC | spend ÷ clicks | Cost of one click |
| CPA | spend ÷ conversions | Cost of one customer / sale |
| Conversion rate | conversions ÷ clicks | How many clickers buy |
| ROAS | revenue ÷ spend | Rs. of revenue per Rs. 1 of ad spend |

Each KPI is calculated from **summed** numbers per group (total clicks ÷ total impressions), not by averaging row-level ratios, which would give small rows the same weight as big ones.

In [ ]:
def kpis(frame, by):
    g = frame.groupby(by, observed=True)[["impressions", "clicks", "spend", "conversions", "revenue"]].sum()
    out = pd.DataFrame({
        "impressions": g.impressions, "clicks": g.clicks, "conversions": g.conversions,
        "spend": g.spend.round(0), "revenue": g.revenue.round(0),
        "CTR_%":  (100 * g.clicks / g.impressions).round(2),
        "CPC":    (g.spend / g.clicks).round(2),
        "CPA":    (g.spend / g.conversions).round(0),
        "CVR_%":  (100 * g.conversions / g.clicks).round(2),
        "ROAS":   (g.revenue / g.spend).round(2),
        "profit_after_ads": (g.revenue - g.spend).round(0),
    })
    return out

overall = kpis(df.assign(all="YouTube total"), "all")
display(overall)

by_campaign = kpis(df, "campaign").sort_values("ROAS", ascending=False)
display(by_campaign)
by_campaign.to_csv("submission/kpis_by_campaign.csv")

for col in ["device", "age", "city", "gender"]:
    t = kpis(df, col).sort_values("ROAS", ascending=False)
    display(t[["spend", "revenue", "CTR_%", "CPC", "CPA", "CVR_%", "ROAS"]])
    t.to_csv(f"submission/kpis_by_{col}.csv")

## Step 6 · Charts

In [ ]:
# Chart 1: ROAS by campaign
ax = by_campaign["ROAS"].sort_values().plot(kind="barh", color="#4C78A8")
plt.axvline(1, color="red", linestyle="--", linewidth=1)
plt.title("ROAS by YouTube Campaign (red line = break-even on ad spend)")
plt.xlabel("ROAS (revenue per Rs. 1 spent)"); plt.ylabel("")
plt.tight_layout(); plt.savefig("submission/chart1_roas_by_campaign.png", dpi=150); plt.show()

# Chart 2: CPA by campaign
by_campaign["CPA"].sort_values(ascending=False).plot(kind="bar", color="#F58518")
plt.title("Cost per Acquisition (CPA) by Campaign")
plt.ylabel("Rs. per conversion"); plt.xlabel(""); plt.xticks(rotation=20, ha="right")
plt.tight_layout(); plt.savefig("submission/chart2_cpa_by_campaign.png", dpi=150); plt.show()

# Chart 3: ROAS by device
kpis(df, "device")["ROAS"].sort_values().plot(kind="barh", color="#54A24B")
plt.title("ROAS by Device"); plt.xlabel("ROAS"); plt.ylabel("")
plt.tight_layout(); plt.savefig("submission/chart3_roas_by_device.png", dpi=150); plt.show()

# Chart 4: weekly spend vs revenue
weekly = df.set_index("date")[["spend", "revenue"]].resample("W").sum()
weekly.plot(marker="o")
plt.title("Weekly Spend vs Revenue"); plt.ylabel("Rs."); plt.xlabel("Week")
plt.tight_layout(); plt.savefig("submission/chart4_weekly_spend_revenue.png", dpi=150); plt.show()

## Step 7 · Marketing Budget Recommendation Engine

**Why not just give everything to the campaign with the highest ROAS?** Because returns **diminish**: the more you push a campaign's daily budget, the more you pay for each extra impression, and the lower the ROAS of the *next* rupee becomes. Retargeting has the best average ROAS, but its audience is small and saturates quickly.

**Method**
1. For each campaign, use the daily history to fit a response curve: `revenue = a × spend^b` (a straight-line fit on log–log scale). `b` below 1 means diminishing returns.
2. Start every campaign at a **minimum 5%** share (keeps brand/awareness campaigns alive) and cap any one campaign at **40%** (avoids over-reliance and extrapolating beyond the history).
3. Hand out the rest of the Rs. 1,000,000 in Rs. 1,000 steps, each time to the campaign whose **next rupee earns the most extra revenue** (the marginal return).
4. Compare the result against simple alternatives: an equal split and repeating last period's split.

The budget is assumed to be spent evenly over the next 30 days.

In [ ]:
BUDGET, HORIZON_DAYS, STEP = 1_000_000, 30, 1_000
MIN_SHARE, MAX_SHARE = 0.05, 0.40

# 1) Fit response curves on daily campaign totals
daily = (df.groupby(["campaign", "date"], observed=True)[["spend", "revenue"]].sum().reset_index())
daily = daily[(daily.spend > 0) & (daily.revenue > 0)]

curves, rows = {}, []
for c, g in daily.groupby("campaign", observed=True):
    b, _ = np.polyfit(np.log(g.spend), np.log(g.revenue), 1)
    b = float(np.clip(b, 0.2, 0.99))                         # keep diminishing returns
    a = float(np.exp(np.mean(np.log(g.revenue) - b * np.log(g.spend))))
    curves[c] = (a, b)
    rows.append({"campaign": c, "a": round(a, 2), "b (diminishing returns)": round(b, 3),
                 "days_in_fit": len(g), "avg_daily_spend": round(g.spend.mean()),
                 "min_daily_spend": round(g.spend.min()), "max_daily_spend": round(g.spend.max())})
curve_table = pd.DataFrame(rows).set_index("campaign")
display(curve_table)

def exp_revenue(c, total_spend):
    """Expected revenue over the horizon if total_spend is spent evenly each day."""
    a, b = curves[c]
    return HORIZON_DAYS * a * (total_spend / HORIZON_DAYS) ** b if total_spend > 0 else 0.0

# 2) Greedy allocation by marginal revenue
names = list(curves)
alloc = {c: MIN_SHARE * BUDGET for c in names}
remaining = BUDGET - sum(alloc.values())
while remaining >= STEP:
    gains = {c: exp_revenue(c, alloc[c] + STEP) - exp_revenue(c, alloc[c])
             for c in names if alloc[c] + STEP <= MAX_SHARE * BUDGET}
    best = max(gains, key=gains.get)
    alloc[best] += STEP
    remaining -= STEP

# 3) Compare strategies
hist_share = df.groupby("campaign", observed=True)["spend"].sum()
hist_share = (hist_share / hist_share.sum()).to_dict()
strategies = {
    "Recommended (marginal return)": alloc,
    "Repeat historical split":       {c: hist_share[c] * BUDGET for c in names},
    "Equal split":                   {c: BUDGET / len(names) for c in names},
}
comparison = pd.DataFrame({
    name: {"expected_revenue": sum(exp_revenue(c, s[c]) for c in names)}
    for name, s in strategies.items()
}).T
comparison["expected_ROAS"] = (comparison.expected_revenue / BUDGET).round(2)
comparison["expected_revenue"] = comparison.expected_revenue.round(0)
display(comparison)
comparison.to_csv("submission/strategy_comparison.csv")

# 4) Final recommendation table
rec = pd.DataFrame({
    "recommended_budget_Rs": pd.Series(alloc).round(0),
    "share_%": (100 * pd.Series(alloc) / BUDGET).round(1),
    "historical_share_%": (100 * pd.Series(hist_share)).round(1),
    "expected_revenue": pd.Series({c: exp_revenue(c, alloc[c]) for c in names}).round(0),
})
rec["expected_ROAS"] = (rec.expected_revenue / rec.recommended_budget_Rs).round(2)
rec = rec.sort_values("recommended_budget_Rs", ascending=False)
display(rec)
rec.to_csv("submission/budget_recommendation.csv")

print("YouTube budget recommendation for the next Rs. 1,000,000:")
for c, r in rec.iterrows():
    print(f"  {c:22s} {r['share_%']:5.1f}%   Rs. {r.recommended_budget_Rs:>9,.0f}")

In [ ]:
# Chart 5: fitted response curves (is spending more still paying off?)
fig, axes = plt.subplots(1, len(names), figsize=(4 * len(names), 3.5), sharey=False)
for ax, c in zip(axes, names):
    g = daily[daily.campaign == c]
    a, b = curves[c]
    xs = np.linspace(g.spend.min(), g.spend.max(), 100)
    ax.scatter(g.spend, g.revenue, s=10, alpha=0.5)
    ax.plot(xs, a * xs ** b, color="red")
    ax.set_title(f"{c}\nb = {b:.2f}", fontsize=9)
    ax.set_xlabel("Daily spend (Rs.)")
axes[0].set_ylabel("Daily revenue (Rs.)")
plt.tight_layout(); plt.savefig("submission/chart5_response_curves.png", dpi=150); plt.show()

# Chart 6: historical vs recommended share
rec[["historical_share_%", "share_%"]].plot(kind="bar", color=["#BBBBBB", "#4C78A8"])
plt.title("Historical vs Recommended Budget Share")
plt.ylabel("% of budget"); plt.xlabel(""); plt.xticks(rotation=20, ha="right")
plt.legend(["Historical", "Recommended"])
plt.tight_layout(); plt.savefig("submission/chart6_budget_shares.png", dpi=150); plt.show()

In [ ]:
# Targeting tip: best / worst audience segments (only segments with enough spend to trust)
seg = (df.groupby(["device", "age", "city"], observed=True)[["spend", "revenue", "conversions"]].sum())
seg = seg[seg.spend >= seg.spend.quantile(0.50)]
seg["ROAS"] = (seg.revenue / seg.spend).round(2)
seg["CPA"]  = (seg.spend / seg.conversions).round(0)
print("Top 5 segments by ROAS:");    display(seg.sort_values("ROAS", ascending=False).head(5)[["spend", "ROAS", "CPA"]])
print("Bottom 5 segments by ROAS:"); display(seg.sort_values("ROAS").head(5)[["spend", "ROAS", "CPA"]])

## Step 8 · Business interpretation

The numbers are printed automatically from the results above. **Rewrite the findings in your own words** and add the actual figures from your run.

In [ ]:
best  = by_campaign.index[0];  worst = by_campaign.index[-1]
print(f"Finding 1: {best} has the highest ROAS ({by_campaign.loc[best, 'ROAS']}) and {worst} the lowest ({by_campaign.loc[worst, 'ROAS']}).")
cheap = by_campaign["CPA"].idxmin()
print(f"Finding 2: {cheap} acquires a customer most cheaply (Rs. {by_campaign.loc[cheap, 'CPA']:,.0f} per conversion).")
dev = kpis(df, "device")["ROAS"]
print(f"Finding 3: {dev.idxmax()} is the best device by ROAS ({dev.max()}), {dev.idxmin()} the weakest ({dev.min()}).")
top = rec.index[0]
print(f"Recommendation: give {top} the largest share ({rec.loc[top, 'share_%']}%). "
      f"Expected ROAS of the plan is {comparison.loc['Recommended (marginal return)', 'expected_ROAS']} vs "
      f"{comparison.loc['Repeat historical split', 'expected_ROAS']} for repeating the historical split.")

**Limitations (keep and expand in your own words)**
- The data is simulated, so the recommendation shows the *method*, not a real-world result.
- The response curves come from only 90 days and are estimates. Some campaigns have few days at high spend, so predictions near the 40% cap are extrapolations.
- ROAS uses revenue, not profit. Margin, product costs and repeat purchases are not modelled.
- The analysis is correlational. Real campaigns should validate the plan with a small test (for example, move 10% of budget and compare results) before committing the full Rs. 1 million.

## Step 9 · Scaling experiment: Pandas vs MapReduce vs Spark

The same job is run three ways at 10,000 → 100,000 → 1,000,000 → 10,000,000 records:

> Read the CSV and calculate total impressions, clicks, spend, conversions and revenue **per campaign**, which gives CTR, CPC, CPA, CVR and ROAS.

- **Pandas:** one machine, whole file in memory.
- **MapReduce:** written by hand in Python: the file is split into chunks, each chunk is **mapped** in a separate process into `(campaign, numbers)` pairs with a local **combiner**, pairs are **shuffled** (grouped by key), then **reduced** (summed). This is the same pattern Hadoop uses, run on this one machine's CPU cores.
- **Spark (PySpark, local mode):** the same job as a DataFrame group-by. Spark's start-up time is measured separately.

Every method must give the same totals, which is checked at the end.

In [ ]:
RUN_10M = True      # needs about 3 GB of RAM and a few minutes; set False if Colab crashes
RUN_SPARK = True
SIZES = [10_000, 100_000, 1_000_000] + ([10_000_000] if RUN_10M else [])
NUM_COLS = ["impressions", "clicks", "spend", "conversions", "revenue"]

# ---------- Pandas ----------
def run_pandas(path):
    t0 = time.time()
    d = pd.read_csv(path, usecols=["campaign"] + NUM_COLS, dtype={"campaign": "category"})
    t_load = time.time() - t0
    t1 = time.time()
    res = d.groupby("campaign", observed=True)[NUM_COLS].sum()
    t_an = time.time() - t1
    mem = d.memory_usage(deep=True).sum() / 1e6
    return res, t_load, t_an, mem

# ---------- MapReduce (hand-written) ----------
def split_file(path, n_chunks):
    """Cut the file into byte ranges that start/end on line boundaries."""
    size = os.path.getsize(path)
    with open(path, "rb") as f:
        header = f.readline().decode().strip().split(",")
        start = f.tell()
        bounds = [start]
        for i in range(1, n_chunks):
            f.seek(start + (size - start) * i // n_chunks)
            f.readline()
            bounds.append(f.tell())
    bounds.append(size)
    return header, [(path, bounds[i], bounds[i + 1]) for i in range(n_chunks)]

def map_task(args):
    """MAP + COMBINE: read one chunk, emit partial sums per campaign."""
    path, lo, hi, idx = args
    partial = defaultdict(lambda: [0, 0, 0.0, 0, 0.0])
    with open(path, "rb") as f:
        f.seek(lo)
        lines = f.read(hi - lo).decode().splitlines()
    ci, ii, ki, si, vi, ri = idx
    for line in lines:
        p = line.split(",")
        acc = partial[p[ci]]
        acc[0] += int(p[ii]); acc[1] += int(p[ki]); acc[2] += float(p[si])
        acc[3] += int(p[vi]); acc[4] += float(p[ri])
    return dict(partial)

def run_mapreduce(path, workers=None):
    workers = workers or os.cpu_count()
    t0 = time.time()
    header, chunks = split_file(path, workers * 4)
    idx = tuple(header.index(c) for c in ["campaign"] + NUM_COLS)
    with Pool(workers) as pool:
        mapped = pool.map(map_task, [(p, lo, hi, idx) for p, lo, hi in chunks])   # MAP
    shuffled = defaultdict(list)                                                    # SHUFFLE
    for part in mapped:
        for key, vals in part.items():
            shuffled[key].append(vals)
    reduced = {k: [sum(v[i] for v in vals) for i in range(5)]                       # REDUCE
               for k, vals in shuffled.items()}
    res = pd.DataFrame.from_dict(reduced, orient="index", columns=NUM_COLS).sort_index()
    return res, time.time() - t0

# ---------- Spark ----------
spark = None
def start_spark():
    global spark
    try:
        import pyspark
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark"])
    from pyspark.sql import SparkSession
    t0 = time.time()
    spark = (SparkSession.builder.master("local[*]").appName("youtube-ads")
             .config("spark.driver.memory", "4g").config("spark.ui.enabled", "false").getOrCreate())
    spark.sparkContext.setLogLevel("ERROR")
    return time.time() - t0

SCHEMA = ("date STRING, campaign STRING, platform STRING, impressions LONG, clicks LONG, spend DOUBLE, "
          "conversions LONG, revenue DOUBLE, age STRING, gender STRING, city STRING, device STRING")

def run_spark(path):
    from pyspark.sql import functions as F
    t0 = time.time()
    sdf = spark.read.csv(path, header=True, schema=SCHEMA)
    out = (sdf.groupBy("campaign").agg(*[F.sum(c).alias(c) for c in NUM_COLS]).toPandas()
              .set_index("campaign").sort_index())
    return out, time.time() - t0      # Spark is lazy, so read + aggregate are timed together

print("Functions ready. Spark is started later, after the Pandas and MapReduce runs (MapReduce uses multiple processes, which is safest before the Spark JVM exists).")

In [ ]:
# Part A: generate each size, run Pandas and MapReduce
results, pd_ref, check_rows = [], {}, []
for n in SIZES:
    big = make_ads_data(n, seed=n)           # fresh simulated data of this size
    path = f"scaling_{n}.csv"
    big.to_csv(path, index=False)
    size_mb = os.path.getsize(path) / 1e6
    del big

    pd_res, p_load, p_an, mem = run_pandas(path)
    mr_res, mr_t = run_mapreduce(path)
    pd_ref[n] = pd_res.sort_index()
    check_rows.append((n, "mapreduce", float((mr_res.revenue - pd_ref[n].revenue).abs().max())))
    results.append({"records": n, "file_MB": round(size_mb, 1),
                    "pandas_load_s": round(p_load, 2), "pandas_analysis_s": round(p_an, 3),
                    "pandas_total_s": round(p_load + p_an, 2), "pandas_memory_MB": round(float(mem), 1),
                    "mapreduce_total_s": round(mr_t, 2)})
    print(results[-1])

# Part B: Spark on the same files
spark_startup = None
if RUN_SPARK:
    spark_startup = start_spark()
    print("Spark start-up seconds:", round(spark_startup, 2))
    for row in results:
        n = row["records"]
        sp_res, sp_t = run_spark(f"scaling_{n}.csv")
        row["spark_total_s"] = round(sp_t, 2)
        check_rows.append((n, "spark", float((sp_res.revenue - pd_ref[n].revenue).abs().max())))
        print(n, "spark", row["spark_total_s"])

for n in SIZES:
    os.remove(f"scaling_{n}.csv")

scaling = pd.DataFrame(results)
scaling.to_csv("submission/scaling_results.csv", index=False)
display(scaling)
print("Spark start-up (one-off) seconds:", spark_startup)
print("Largest revenue difference vs Pandas, by size and method (should be ~0):")
for n, m, diff in check_rows:
    print(f"  {n:>10,} {m:10s} {diff:.6f}")

In [ ]:
# Chart 7: total time by method (log scale because sizes grow 10x each step)
cols = [c for c in ["pandas_total_s", "mapreduce_total_s", "spark_total_s"] if c in scaling.columns]
ax = scaling.set_index("records")[cols].plot(marker="o")
ax.set_xscale("log"); ax.set_yscale("log")
plt.title("Time to Calculate Campaign KPIs (read + aggregate)")
plt.xlabel("Number of records"); plt.ylabel("Seconds")
plt.legend([c.replace("_total_s", "") for c in cols])
plt.tight_layout(); plt.savefig("submission/chart7_scaling_comparison.png", dpi=150); plt.show()

**Scaling experiment results: write these in your own words from the table above**

Suggested points to cover (use your actual numbers):
- Which method was fastest at 10,000 records, and which at 10,000,000? Does the order change?
- Spark has a fixed start-up cost (shown above). At what size does that cost start to pay off, or does it not within the tested sizes?
- How did Pandas memory grow with records, and what size would break a typical laptop?
- The MapReduce here runs on one machine's cores, so it shows the *pattern*. On a real cluster the map tasks would run on many machines. Explain what this simulation can and cannot show.
- Remember this is a workload test on simulated data, not new evidence about advertising performance.

## Step 10 · Package the submission

In [ ]:
summary = pd.DataFrame({
    "item": ["Rows (raw)", "Rows (after cleaning)", "Columns", "Missing cells (after cleaning)",
             "Duplicate rows (raw)", "Platform", "Date range (days)", "Budget optimised (Rs.)"],
    "value": [len(raw), len(df), df.shape[1], int(df.isna().sum().sum()),
              int(raw.duplicated().sum()), "YouTube", N_DAYS, BUDGET],
})
summary.to_csv("submission/dataset_summary.csv", index=False)
df.to_csv("submission/cleaned_data.csv", index=False)
display(summary)

shutil.make_archive("project2_submission", "zip", "submission")
try:
    from google.colab import files
    files.download("project2_submission.zip")
except ImportError:
    print("Zip created: project2_submission.zip")